In [ ]:
from pprint import pprint
from IPython.display import display, Markdown

MODE = "replay"

INPUT_CASE = {
    "demo_id": "demo_f5_observability_first",
    "demo_version": "v1",
    "case_id": "f5_observability_case_001",
    "title": "Workflow failure with too little visibility for safe diagnosis",
    "task_type": "workflow_debugging",
    "family_target": {
        "primary_family": "F5",
        "secondary_family": "F4",
        "best_current_fit": "F5_N01 Failure Path Opacity",
        "broken_invariant": "failure_path_visibility_broken"
    },
    "user_question": "Why is the workflow returning irrelevant answers, and what should be fixed first?",
    "thin_trace": [
        "Step 1: User query received",
        "Step 2: Retrieval executed",
        "Step 3: Final answer produced",
        "Observed symptom: answer is irrelevant to the user question"
    ],
    "observability_uplift": {
        "retrieval_trace": [
            "retriever_query = 'general company summary'",
            "top_k = 2",
            "returned_chunk_ids = ['chunk_014', 'chunk_019']",
            "both chunks are broad product overviews, not release-note evidence"
        ],
        "candidate_trace": [
            "candidate_answer_1 = 'The workflow likely needs a stronger generation step'",
            "candidate_answer_2 = 'The retrieval target appears off-topic relative to the question'"
        ],
        "post_check_trace": [
            "answer_to_question_alignment = low",
            "evidence_to_answer_alignment = low",
            "retrieval_to_question_alignment = low"
        ]
    }
}

REPLAY_OUTPUTS = {
    "baseline_diagnosis": "The workflow likely needs a stronger final prompt or a better answer generation step. Try improving the model instructions first.",
    "baseline_problem": "The diagnosis jumps too early to a direct fix even though the workflow is still too opaque for a safe root-cause claim.",
    "repaired_diagnosis": "The first repair move should be observability uplift. The workflow should not be treated as execution-first or reasoning-first yet, because the visible trace is still too thin. Once retrieval trace, candidate trace, and post-check trace are exposed, the system becomes diagnosable and the off-target retrieval signal becomes inspectable.",
    "before_state": "opaque",
    "after_state": "diagnosable"
}

EXPECTED_OUTPUT = {
    "primary_family": "F5",
    "secondary_family": "F4",
    "best_current_fit": "F5_N01 Failure Path Opacity",
    "broken_invariant": "failure_path_visibility_broken",
    "first_repair_move": "observability insertion"
}

baseline_prompt = f"""
You are a workflow debugging assistant.

A system received a user query, ran retrieval, and produced a final answer.
The final answer was irrelevant to the user question.

Available trace:
- {INPUT_CASE["thin_trace"][0]}
- {INPUT_CASE["thin_trace"][1]}
- {INPUT_CASE["thin_trace"][2]}
- {INPUT_CASE["thin_trace"][3]}

Explain what probably went wrong.
Then recommend one direct fix to apply immediately.

Assume the current trace is enough.
Keep the answer short and confident.
""".strip()

repaired_prompt = f"""
You are diagnosing a workflow failure.

Question:
{INPUT_CASE["user_question"]}

Thin trace:
- {INPUT_CASE["thin_trace"][0]}
- {INPUT_CASE["thin_trace"][1]}
- {INPUT_CASE["thin_trace"][2]}
- {INPUT_CASE["thin_trace"][3]}

Additional observability:
Retrieval trace:
- {INPUT_CASE["observability_uplift"]["retrieval_trace"][0]}
- {INPUT_CASE["observability_uplift"]["retrieval_trace"][1]}
- {INPUT_CASE["observability_uplift"]["retrieval_trace"][2]}
- {INPUT_CASE["observability_uplift"]["retrieval_trace"][3]}

Candidate trace:
- {INPUT_CASE["observability_uplift"]["candidate_trace"][0]}
- {INPUT_CASE["observability_uplift"]["candidate_trace"][1]}

Post-check trace:
- {INPUT_CASE["observability_uplift"]["post_check_trace"][0]}
- {INPUT_CASE["observability_uplift"]["post_check_trace"][1]}
- {INPUT_CASE["observability_uplift"]["post_check_trace"][2]}

Answer in this order:
1. What the first failure family is
2. Why F5 is a better first cut than F4
3. What the first repair move should be
4. What the visible evidence now suggests
""".strip()

def section(title: str):
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)

def bullet_list(items):
    for item in items:
        print(f"- {item}")

display(Markdown("""
# Problem Map 3.0 Troubleshooting Atlas
## Demo 2 · F5 Observability First

### What this experiment is
This notebook is a **replay-only MVP** experiment.

It is designed to show that some failures should not be repaired by guessing a root cause too early.
Some failures first break at **Observability & Diagnosability Integrity**, so the correct first move is to expose the failure path before attempting deeper repair.

### Why this notebook is replay-only
For this MVP, **live mode is not required**.
The point of this demo is not model creativity.
The point is to make the **before / after visibility shift** obvious and easy to inspect.

### What you should expect to see
- The baseline overcommits under thin trace
- The repaired version does not jump too early
- The repaired version treats **observability uplift** as the correct first move
- The workflow shifts from **opaque** to **diagnosable**
"""))

section("Mode")
print(f"MODE = {MODE}")

section("Case overview")
print("Title:")
print(INPUT_CASE["title"])
print()
print("Question:")
print(INPUT_CASE["user_question"])

section("Thin trace")
bullet_list(INPUT_CASE["thin_trace"])

section("Atlas routing target")
pprint(INPUT_CASE["family_target"])

section("Baseline prompt")
print(baseline_prompt)

section("Repaired prompt")
print(repaired_prompt)

section("Replay mode · baseline")
print("Baseline diagnosis:")
print(REPLAY_OUTPUTS["baseline_diagnosis"])
print()
print("Why baseline is weak:")
print(REPLAY_OUTPUTS["baseline_problem"])

section("Replay mode · repaired")
print("Repaired diagnosis:")
print(REPLAY_OUTPUTS["repaired_diagnosis"])

section("Replay mode · state shift")
print("Before:", REPLAY_OUTPUTS["before_state"])
print("After :", REPLAY_OUTPUTS["after_state"])

section("Expected effect checklist")
print("1. Baseline treats a thin trace as if it were sufficient.")
print("2. Baseline jumps too early to a direct repair move.")
print("3. Repaired version identifies F5 before F4.")
print("4. Repaired version treats observability uplift as the first repair move.")
print("5. The main gain is diagnosability, not a magically perfect final answer.")

section("Expected success contract")
pprint(EXPECTED_OUTPUT)

display(Markdown("""
## Back to the main page

Read the full product page here:
[Problem Map 3.0 Troubleshooting Atlas](https://github.com/onestardao/WFGY/blob/main/ProblemMap/wfgy-ai-problem-map-troubleshooting-atlas.md)

If you like the project, star the repo ⭐
"""))